This document analyzes the dataset logged by CARLA's system. The metrics to be measured are the following: 
1. **Reaction Time**
    - This is from the time the alert was sent until the time the driver made an action. This will help the researches know the appropriate time to give an alert. 
2. **Completion Time**
    - This measures the duration of the whole violation. From the alert being triggered until the speeding gets resolved. 
3. **Ignored Alert** 
    - This checks whether the driver made an action to the alert or ignored it. This will be used to prove the effectiveness of providing an alert to decrease speed.
4. **Speed Change**
    - This checks the speed. 
5. **Alert Effectiveness**
    - This computes for the number of resolved speeding violations, where the rider either *reduced throttle* or *brakes* which is then divided by all violations. (resolved/all violations) 

*Note: An action made by the driver can be either a release of the throttle or pressing on the brakes.* 

In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import chi2_contingency
from statsmodels.stats.anova import AnovaRM
from scipy.stats import ttest_rel

In [2]:
df = pd.read_csv("cleaned_Sim.csv")

df.head()


,timestamp,participant_no,phase,scenario,speed_kmh,event,details,speed_limit,Location_X,Location_Y,overspeed,reduce_throttle,resolved,event_id
0,1.771924e+09,17,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,NaN,NaN,False,False,False,0
1,1.771924e+09,17,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,NaN,NaN,False,False,False,0
2,1.771924e+09,17,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,NaN,NaN,False,False,False,0
3,1.771924e+09,17,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,...",30.0,NaN,NaN,False,False,False,0
4,1.771924e+09,17,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,NaN,NaN,False,False,False,0


In [3]:
df.columns

Index(['timestamp', 'participant_no', 'phase', 'scenario', 'speed_kmh',
       'event', 'details', 'speed_limit', 'Location_X', 'Location_Y',
       'overspeed', 'reduce_throttle', 'resolved', 'event_id'],
      dtype='object')

In [4]:
df["scenario"].unique()

array(['WEBSOCKET', 'TRAFFIC_LIGHT', 'SPEED_LIMIT', 'STOP'], dtype=object)

In [5]:
df["details"].str.contains("TrafficLight", na=False).sum()

276

In [6]:
df.loc[
    (df["event_id"]==45),
    ["speed_kmh", "scenario", "speed_limit", "details", "event_id"]
]

,speed_kmh,scenario,speed_limit,details,event_id
77952,37.25,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 37.25, ""speed_limit"": 30...",45
77953,37.55,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 37.55, ""speed_limit"": 30...",45
77954,38.12,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.12, ""speed_limit"": 30...",45
77955,38.12,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.12, ""speed_limit"": 30...",45
77956,38.12,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.12, ""speed_limit"": 30...",45
77957,38.52,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 38.52, ""speed_limit"": 30...",45
77958,39.81,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 39.81, ""speed_limit"": 30...",45
77959,40.03,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 40.03, ""speed_limit"": 30...",45
77960,40.03,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 40.03, ""speed_limit"": 30...",45
77961,40.03,WEBSOCKET,30.0,"{""phase"": 3, ""speed"": 40.03, ""speed_limit"": 30...",45


In [7]:
df.loc[
    df["event_id"].isin([183]),
    ["phase", "speed_kmh", "speed_limit", "details", "event_id"]
]

,phase,speed_kmh,speed_limit,details,event_id
137610,3,70.86,60.0,"{""phase"": 3, ""speed"": 70.86, ""speed_limit"": 60...",183
137611,3,70.98,60.0,"{""phase"": 3, ""speed"": 70.98, ""speed_limit"": 60...",183
137612,3,70.98,60.0,"{""phase"": 3, ""speed"": 70.98, ""speed_limit"": 60...",183
137613,3,70.98,60.0,"{""phase"": 3, ""speed"": 70.98, ""speed_limit"": 60...",183
137614,3,71.34,60.0,"{""phase"": 3, ""speed"": 71.34, ""speed_limit"": 60...",183
137615,3,71.45,60.0,"{""phase"": 3, ""speed"": 71.45, ""speed_limit"": 60...",183
137616,3,71.45,60.0,"{""phase"": 3, ""speed"": 71.45, ""speed_limit"": 60...",183
137617,3,71.68,60.0,"{""phase"": 3, ""speed"": 71.68, ""speed_limit"": 60...",183
137618,3,71.79,60.0,"{""phase"": 3, ""speed"": 71.79, ""speed_limit"": 60...",183
137619,3,71.88,60.0,"{""phase"": 3, ""speed"": 71.88, ""speed_limit"": 60...",183


In [8]:
df.loc[df["speed_limit"] == 60, "event_id"].unique()

array([  5,   0,  29,  51, 122, 136, 183, 235, 250, 270, 289, 290, 291,
       292, 301, 332, 412, 422, 432, 522, 532])

## Metrics Computation & Analysis

In [9]:
df_Metrics = df.copy()

### Reaction Time

*action=REDUCE_THROTTLE* is a signal that the driver released yung tapak niya sa pedal. 

In [10]:
df.loc[
    df["event_id"].isin([9]),
    ["phase","speed_kmh", "scenario", "speed_limit", "details", "event_id"]
]

,phase,speed_kmh,scenario,speed_limit,details,event_id
15607,2,40.00,SPEED_LIMIT,NaN,action=REDUCE_THROTTLE|time=0.52s|control={'th...,9
15608,2,40.10,SPEED_LIMIT,30.0,Speed 40.10 km/h > limit 30.00 km/h at Locatio...,9
15609,2,36.87,SPEED_LIMIT,NaN,action=VIOLATION_RESOLVED|time=0.99s|reaction_...,9


In [11]:
df_Metrics["carla_reaction_time"] = (
    df_Metrics.loc[df_Metrics["scenario"] == "SPEED_LIMIT", "details"]
    .str.extract(r"action=REDUCE_THROTTLE\|time=(\d+\.\d+)s")
    .astype(float)
)

In [12]:
carla_reactions = df_Metrics.loc[
    df_Metrics["carla_reaction_time"].notna(),
    ["event_id", "carla_reaction_time"]
]

In [13]:
display(carla_reactions.head(5))

,event_id,carla_reaction_time
767,1,0.76
5496,2,1.86
6252,3,0.76
7122,4,0.21
15598,7,1.92


*Note: in the cases of NaN values: at times the rider is just above the speeding threshold, the system in CARLA might not have detected enough throttle release. Therefore, in the subtracting the action to the start becomes not applicable.*

### Completion Time

In [14]:
df_Metrics["carla_completion_time"] = df_Metrics["details"].str.extract(
    r"action=VIOLATION_RESOLVED\|time=(\d+\.\d+)s"
).astype(float)

In [15]:
carla_complete = df_Metrics.loc[
    df_Metrics["carla_completion_time"].notna(),
    ["event_id", "carla_completion_time"]
]

In [16]:
display(carla_complete.tail(10))

,event_id,carla_completion_time
348702,0,0.15
349962,534,0.49
350409,535,3.98
351085,536,0.22
351980,537,2.39
352149,538,0.29
352709,539,1.77
353765,540,3.45
355066,541,1.91
356023,542,1.77


### Ignored Alert

In [17]:
df_Metrics["alert_ignored"] = (df_Metrics["carla_reaction_time"] >= 2.5).fillna(0).astype(int)

In [18]:
ignore = df_Metrics.loc[
    df_Metrics["carla_reaction_time"].notna(),  # only rows with reaction time
    ["event_id", "carla_reaction_time", "alert_ignored"]  # columns to display
]

ignore.tail(10)

,event_id,carla_reaction_time,alert_ignored
348692,532,0.24,0
348698,533,0.57,0
349958,534,0.37,0
350380,535,2.55,1
351948,537,0.82,0
352144,538,0.12,0
352690,539,0.89,0
353744,540,2.47,0
355047,541,1.03,0
356003,542,0.87,0


### Alert Effectiveness

In [19]:
speedingCount = df_Metrics["details"].str.contains("VIOLATION_RESOLVED", na=False).sum() # dapat not ignored to
unignored = (carla_reactions["carla_reaction_time"] < 2.5).sum()
ignored = (carla_reactions["carla_reaction_time"] >= 2.5).sum()
valid_reactions = (carla_reactions["carla_reaction_time"]).notna().sum()
valid_complete = (carla_complete["carla_completion_time"].notna().sum())

In [20]:
total_effective = unignored/speedingCount

In [21]:
print("Unignored Count: ", unignored)
print("Ignored Count: ", ignored)
print("Supposed Total events: ", unignored + ignored)
print("Reaction Time: ", valid_reactions)
print("Completion Time: ", valid_complete)
print("Total resolved violations: ", speedingCount)
print(f"Effectiveness: {total_effective*100:.4f}%")

Unignored Count:  435
Ignored Count:  29
Supposed Total events:  464
Reaction Time:  464
Completion Time:  601
Total resolved violations:  601
Effectiveness: 72.3794%


In [22]:
ignored_result=(ignored / df_Metrics["event_id"].nunique())

In [23]:
print(f"Ignored alerts: {ignored_result* 100:.4f}%")

Ignored alerts: 5.3407%


In [24]:
summary_table = pd.DataFrame({
    "Mean": [
        carla_complete["carla_completion_time"].mean(),
        carla_reactions["carla_reaction_time"].mean()
    ],
    "Median": [
        carla_complete["carla_completion_time"].median(),
        carla_reactions["carla_reaction_time"].median()
    ],
    "Std Dev": [
        carla_complete["carla_completion_time"].std(),
        carla_reactions["carla_reaction_time"].std()
    ]
},
index=[
    "Completion Time",
    "Reaction Time"
])

print(summary_table)

                     Mean  Median   Std Dev
Completion Time  2.499850   1.710  2.596440
Reaction Time    1.061595   0.835  0.766071


## Alert-Phase Analysis

| Metric                     | Why                             |
| -------------------------- | ------------------------------- |
| Mean Reaction Time         | how quickly riders respond      |
| SD Reaction Time           | variability                     |
| Mean Completion Time       | how fast violation is corrected |
| Ignored Alert Rate         | how often alerts are ignored    |



In [25]:
results = []

for phase, df_phase in df_Metrics.groupby("phase"):

    n_events = df_phase["carla_reaction_time"].notna().sum()

    reactions = df_phase["carla_reaction_time"].dropna()
    mean_reaction = reactions.mean()
    median_reactions = reactions.median()
    sd_reaction = reactions.std()

    results.append({
        "Phase": phase,
        "n_events": n_events,
        "Mean ReactionT": mean_reaction,
        "Median ReactionT": median_reactions,
        "SD ReactionT": sd_reaction
    })

metrics_table = pd.DataFrame(results).sort_values("Phase")
display(metrics_table)

,Phase,n_events,Mean ReactionT,Median ReactionT,SD ReactionT
0,1,112,1.179821,1.04,0.776053
1,2,90,1.145333,0.94,0.821778
2,3,146,0.964932,0.75,0.747430
3,4,116,1.004138,0.78,0.721209


In [26]:
results = []

for phase, df_phase in df_Metrics.groupby("phase"):

    n_events = df_phase["details"].str.contains("VIOLATION_RESOLVED", na=False).sum()

    complete = df_phase["carla_completion_time"].dropna()
    mean_complete = complete.mean()
    sd_complete = complete.std()

    total_violations = n_events
    unignored = (df_phase["carla_reaction_time"] < 2.5).sum()

    ignored_percent = ((total_violations - unignored) / total_violations) * 100 if total_violations > 0 else 0
    effectiveness = (unignored / total_violations)*100 if total_violations > 0 else 0

    results.append({
        "Phase": phase,
        "n_events": n_events,
        "Mean Complete": mean_complete,
        "STD Complete" : sd_complete,
        "Ignored %": ignored_percent,
        "Effectiveness": effectiveness
    })

metrics_table = pd.DataFrame(results).sort_values("Phase")
display(metrics_table)

,Phase,n_events,Mean Complete,STD Complete,Ignored %,Effectiveness
0,1,156,2.731410,2.325275,32.692308,67.307692
1,2,122,2.893689,3.308951,32.786885,67.213115
2,3,182,2.117088,2.194470,24.725275,75.274725
3,4,141,2.396950,2.611643,21.276596,78.723404


The baseline 

In [27]:
df_clean = df_Metrics.dropna(subset=["carla_completion_time"])

In [28]:
groups = [group["carla_completion_time"].values
          for name, group in df_clean.groupby("phase")]

f_statistic, p_value = stats.f_oneway(*groups)

print("F-statistic:", f_statistic)
print("p-value:", p_value)

F-statistic: 2.765690541907601
p-value: 0.041176577017671114


In [29]:
alerts = df_Metrics[df_Metrics["carla_reaction_time"].notna()]

ignored_alerts = (alerts["carla_reaction_time"] >= 2.5).sum()

overall_ignored_percent = ignored_alerts / len(alerts) * 100

print(f"Overall ignored alerts: {overall_ignored_percent:.2f}%")

Overall ignored alerts: 6.25%


In [30]:
df_clean.groupby("phase")["carla_completion_time"].std()

phase
1    2.325275
2    3.308951
3    2.194470
4    2.611643
Name: carla_completion_time, dtype: float64

In [31]:
df_clean.groupby("phase")["carla_completion_time"].mean()

phase
1    2.731410
2    2.893689
3    2.117088
4    2.396950
Name: carla_completion_time, dtype: float64

In [32]:
# collapse to one row per event
alerts = df_Metrics.groupby("event_id").agg({
    "phase": "first",           # each event belongs to one phase
    "alert_ignored": "max"      # if any frame was ignored, mark the event as ignored
}).reset_index()

# now build the correct contingency table
contingency_table = pd.crosstab(alerts["phase"], alerts["alert_ignored"])
print(contingency_table)

alert_ignored    0  1
phase                
1              105  7
2              111  8
3              163  9
4              135  5


In [33]:
df_Metrics.groupby("event_id").size().sort_values(ascending=False).head(5)

event_id
0      337156
299       492
75        372
481       368
43        355
dtype: int64

In [34]:
results = []

for phase, df_phase in df.groupby("phase"):

    # total traffic lights encountered
    lights_encountered = df_phase["scenario"].eq("TRAFFIC_LIGHT").sum()

    # red light violations
    violations = df_phase["event"].str.contains("RED_LIGHT_VIOLATION", na=False).sum()

    # compliance rate
    compliance_rate = ((lights_encountered - violations) / lights_encountered) if lights_encountered > 0 else 0

    results.append({
        "Phase": phase,
        "Traffic Lights Encountered": lights_encountered,
        "Violations": violations,
        "Compliance Rate": compliance_rate
    })

traffic_table = pd.DataFrame(results).sort_values("Phase")

display(traffic_table)

,Phase,Traffic Lights Encountered,Violations,Compliance Rate
0,1,108,56,0.481481
1,2,125,55,0.560000
2,3,177,86,0.514124
3,4,167,79,0.526946


In [35]:
# ground truth encounters based on route design
route_lights = {
    1: 7,
    2: 7,
    3: 11,
    4: 8
}

participants = 20

traffic_table["Traffic Lights Encountered"] = (
    traffic_table["Phase"].map(route_lights) * participants
)

# recompute compliance
traffic_table["Compliance Rate"] = (
    (traffic_table["Traffic Lights Encountered"] - traffic_table["Violations"])
    / traffic_table["Traffic Lights Encountered"]
)

display(traffic_table)

,Phase,Traffic Lights Encountered,Violations,Compliance Rate
0,1,140,56,0.600000
1,2,140,55,0.607143
2,3,220,86,0.609091
3,4,160,79,0.506250


In [36]:
resolved_rows = df_Metrics[df_Metrics["details"].str.contains("REDUCE_THROTTLE", na=False)]
resolved_rows.tail(5)

,timestamp,participant_no,phase,scenario,speed_kmh,event,details,speed_limit,Location_X,Location_Y,overspeed,reduce_throttle,resolved,event_id,carla_reaction_time,carla_completion_time,alert_ignored
352144,1.771681e+09,11,3,SPEED_LIMIT,37.26,REACTION,action=REDUCE_THROTTLE|time=0.12s|control={'th...,NaN,NaN,NaN,False,True,False,538,0.12,NaN,0
352690,1.771681e+09,11,3,SPEED_LIMIT,44.97,REACTION,action=REDUCE_THROTTLE|time=0.89s|control={'th...,NaN,NaN,NaN,False,True,False,539,0.89,NaN,0
353744,1.771681e+09,11,3,SPEED_LIMIT,48.03,REACTION,action=REDUCE_THROTTLE|time=2.47s|control={'th...,NaN,NaN,NaN,False,True,False,540,2.47,NaN,0
355047,1.771682e+09,11,3,SPEED_LIMIT,42.17,REACTION,action=REDUCE_THROTTLE|time=1.03s|control={'th...,NaN,NaN,NaN,False,True,False,541,1.03,NaN,0
356003,1.771682e+09,11,3,SPEED_LIMIT,49.72,REACTION,action=REDUCE_THROTTLE|time=0.87s|control={'th...,NaN,NaN,NaN,False,True,False,542,0.87,NaN,0


In [37]:
df.loc[
    df["details"].str.contains("VIOLATION_RESOLVED", na=False),
    ["scenario", "speed_kmh", "speed_limit", "event", "details", "event_id"]
].head()

,scenario,speed_kmh,speed_limit,event,details,event_id
780,SPEED_LIMIT,36.30,NaN,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,1
5506,SPEED_LIMIT,36.39,NaN,REACTION,action=VIOLATION_RESOLVED|time=2.30s|reaction_...,2
6265,SPEED_LIMIT,35.52,NaN,REACTION,action=VIOLATION_RESOLVED|time=1.32s|reaction_...,3
7124,SPEED_LIMIT,36.53,NaN,REACTION,action=VIOLATION_RESOLVED|time=0.26s|reaction_...,4
7131,SPEED_LIMIT,36.93,NaN,REACTION,action=VIOLATION_RESOLVED|time=5.16s|reaction_...,0


In [38]:
df_Metrics.columns

Index(['timestamp', 'participant_no', 'phase', 'scenario', 'speed_kmh',
       'event', 'details', 'speed_limit', 'Location_X', 'Location_Y',
       'overspeed', 'reduce_throttle', 'resolved', 'event_id',
       'carla_reaction_time', 'carla_completion_time', 'alert_ignored'],
      dtype='object')